# How to simulate on a GPU with CUDA

ffsim can run state vector simulations on NVIDIA GPUs using [CuPy](https://cupy.dev/). GPU support covers the circuit simulation path: the gate application functions, `apply_unitary` (including unitary cluster Jastrow operators), and Trotter time evolution. In addition, the linear operator returned by `ffsim.linear_operator` for a molecular Hamiltonian performs its matrix-vector products on the GPU automatically.

**Note:** Unlike the other guides, this one requires a CUDA-capable GPU, so it is not executed when the documentation is built. The outputs shown below were produced on an NVIDIA Quadro GP100.

## Installation

GPU support requires CuPy, which is an optional dependency. Install ffsim with the extra matching your CUDA version:

```bash
pip install "ffsim[cuda12]"  # CUDA 12.x
pip install "ffsim[cuda13]"  # CUDA 13.x
```

or install the corresponding CuPy package (`cupy-cuda12x` or `cupy-cuda13x`) directly. See the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html) for details. Note that your GPU must be supported by the CUDA version you choose; for example, CUDA 13 dropped support for Maxwell, Pascal, and Volta GPUs, so those require the `cuda12` extra.

## Moving a state vector to the GPU

To run a simulation on the GPU, convert the state vector to a CuPy array with `cupy.asarray` and pass it to ffsim functions as usual. Functions dispatch on the array type: given a CuPy array, they execute CUDA kernels on the GPU and return a CuPy array. Use `cupy.asnumpy` to transfer the result back to the CPU.

Only the state vector lives on the GPU. Gate parameters, such as orbital rotation matrices and Hamiltonian objects, remain Numpy arrays. State vectors must have dtype `complex128` and be contiguous, which is what `cupy.asarray` produces from a state vector created by ffsim.

Let's create a Hartree-Fock state and transfer it to the GPU.

In [1]:
import cupy
import numpy as np

import ffsim

norb = 14
nelec = (7, 7)

rng = np.random.default_rng(1234)
op = ffsim.random.random_ucj_op_spin_balanced(norb, n_reps=2, seed=rng)

# Create the initial state on the CPU and transfer it to the GPU
vec = ffsim.hartree_fock_state(norb, nelec)
vec_gpu = cupy.asarray(vec)

print(f"Dimension of the vector space: {ffsim.dim(norb, nelec)}")
print(f"Type of the state vector: {type(vec_gpu)}")
print(f"Data type of the state vector: {vec_gpu.dtype}")

Dimension of the vector space: 11778624
Type of the state vector: <class 'cupy.ndarray'>
Data type of the state vector: complex128


Now let's apply the unitary cluster Jastrow operator. Because the input is a CuPy array, the simulation runs on the GPU, and the result is also a CuPy array.

In [2]:
result_gpu = ffsim.apply_unitary(vec_gpu, op, norb=norb, nelec=nelec)

print(f"Type of the result: {type(result_gpu)}")

# Transfer the final state back to the CPU
result = cupy.asnumpy(result_gpu)

Type of the result: <class 'cupy.ndarray'>


## Checking the result against the CPU

The GPU implementation computes the same thing as the CPU implementation, up to floating point rounding. Let's verify that by running the same simulation with the Numpy state vector.

In [3]:
result_cpu = ffsim.apply_unitary(vec, op, norb=norb, nelec=nelec)

np.testing.assert_allclose(result, result_cpu, atol=1e-12)

print(f"Maximum difference: {np.max(np.abs(result - result_cpu))}")

Maximum difference: 2.9592960165596885e-18


Transferring data between the host and the device is not free, and for large state vectors it can cost as much as a few gates. Rather than moving the state vector back and forth, construct it on the device, apply the entire circuit there, and transfer the result back once at the end.

## Trotter simulation of a molecular Hamiltonian

The Trotter simulation functions accept GPU state vectors, too. Let's build a nitrogen molecule to use as an example, following [Simulate Trotterized time evolution for the molecular Hamiltonian](simulate-trotter-mol-ham.ipynb).

In [4]:
import pyscf

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = pyscf.data.elements.chemcore(mol)
active_space = range(n_frozen, 14)

# Get molecular data and Hamiltonian
scf = pyscf.scf.RHF(mol).run()
mol_data = ffsim.MolecularData.from_scf(scf, active_space=active_space)
norb, nelec = mol_data.norb, mol_data.nelec
mol_hamiltonian = mol_data.hamiltonian

# Get the Hamiltonian in double-factorized form
df_hamiltonian = ffsim.DoubleFactorizedHamiltonian.from_molecular_hamiltonian(
    mol_hamiltonian
)

print(f"norb = {norb}")
print(f"nelec = {nelec}")
print(f"Dimension of the vector space: {ffsim.dim(norb, nelec)}")

converged SCF energy = -108.835236570774
norb = 12
nelec = (5, 5)
Dimension of the vector space: 627264


As before, only the state vector is transferred to the GPU. The double-factorized Hamiltonian stays on the host.

In [5]:
# Construct the initial state on the GPU
initial_state = ffsim.hartree_fock_state(norb, nelec)
initial_state_gpu = cupy.asarray(initial_state)

# Set the evolution time
evolution_time = 1.0

# Perform the Trotterized time evolution on the GPU
final_state_gpu = ffsim.simulate_trotter_double_factorized(
    initial_state_gpu,
    df_hamiltonian,
    evolution_time,
    norb=norb,
    nelec=nelec,
    n_steps=5,
    order=1,
)

print(f"Type of the result: {type(final_state_gpu)}")

Type of the result: <class 'cupy.ndarray'>


## Time evolution with a linear operator

The linear operator returned by `ffsim.linear_operator` for a `MolecularHamiltonian` (or `MolecularHamiltonianSpinless`) with real-valued tensors performs its matrix-vector products on the GPU when CuPy and a CUDA device are available. Unlike the gate functions, this linear operator takes and returns Numpy arrays, so code that uses it speeds up without any changes. The GPU is not used for Hamiltonians with complex-valued tensors, or when the computation does not fit in GPU memory; in those cases the computation falls back to the CPU.

For example, computing exact time evolution with `scipy.sparse.linalg.expm_multiply` uses the GPU automatically. Let's use it to check the error from Trotterization. Note that the Trotter-evolved state has to be transferred back to the CPU first.

In [6]:
import scipy.sparse.linalg

# Convert the Hamiltonian to a LinearOperator
linop = ffsim.linear_operator(mol_hamiltonian, norb=norb, nelec=nelec)

# Compute the exact result of time evolution
exact_state = scipy.sparse.linalg.expm_multiply(
    -1j * evolution_time * linop,
    initial_state,
    traceA=-1j * evolution_time * ffsim.trace(mol_hamiltonian, norb=norb, nelec=nelec),
)

# Compute fidelity between results from Trotterized evolution and exact evolution
fidelity = abs(np.vdot(cupy.asnumpy(final_state_gpu), exact_state))

print(f"Fidelity of Trotter-evolved state with exact state: {fidelity}")

Fidelity of Trotter-evolved state with exact state: 0.99994056475516


## Supported operations

The following operations accept CuPy state vectors:

- `ffsim.apply_orbital_rotation`
- `ffsim.apply_num_op_sum_evolution`
- `ffsim.apply_diag_coulomb_evolution` (both the standard and Z representations)
- `ffsim.apply_quad_ham_evolution`
- The basic gates: `ffsim.apply_givens_rotation`, `ffsim.apply_tunneling_interaction`, `ffsim.apply_num_interaction`, `ffsim.apply_num_num_interaction`, `ffsim.apply_on_site_interaction`, `ffsim.apply_num_op_prod_interaction`, `ffsim.apply_hop_gate`, `ffsim.apply_fsim_gate`, and `ffsim.apply_fswap_gate`
- `ffsim.apply_unitary` with any operator composed of the above, including the unitary cluster Jastrow (UCJ) operators
- Trotter simulation: `ffsim.simulate_trotter_diag_coulomb_split_op` and `ffsim.simulate_trotter_double_factorized`

## Operations that run on the CPU

Other operations, such as other operator contractions, expectation values, reduced density matrices, sampling, and the Qiskit integration, currently run only on the CPU. Transfer the state vector back with `cupy.asnumpy` to use them.

In [7]:
from collections import Counter

# Transfer the final state back to the CPU
final_state = cupy.asnumpy(final_state_gpu)

samples = ffsim.sample_state_vector(
    final_state, norb=norb, nelec=nelec, shots=100, seed=rng
)

for bitstring, count in Counter(samples).most_common(5):
    print(f"{bitstring} {count}")

000000011111000000011111 88
000001011011000001011011 3
000100110101000000011111 1
000001011101000001011101 1
010000010111000100001111 1
